# Quickstart — TimesFM + Schaake Shuffle

Este notebook sirve como punto de entrada para una persona que utiliza el repositorio por primera vez.

La generación de escenarios **no se reimplementa aquí**. La función principal llama al mismo módulo utilizado por la CLI:

```text
wind_scenarios.cli.generate
```

y luego utiliza:

```text
wind_scenarios.evaluation
```

para exportar el `ground_truth.csv` **después** de completar la generación.

De esta forma se mantiene explícita la separación:

```text
datos conocidos -> TimesFM -> Schaake Shuffle -> escenarios
                                                 |
                                                 v
                                   evaluación ex-post
                                   ground_truth.csv
```

## 1. Preparar el entorno

Desde PowerShell, en la raíz del repositorio:

```powershell
uv sync
```

Para abrir Jupyter Lab sin agregar Jupyter permanentemente al proyecto:

```powershell
uv run --with jupyter jupyter lab
```

El dataset debe estar disponible en:

```text
data/wind_ab_daily.csv
```

con columnas:

```text
DATE,WIND
```

In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import pandas as pd

from wind_scenarios.data import (
    DEFAULT_DATA_PATH,
    PROJECT_ROOT,
)
from wind_scenarios.evaluation import (
    load_ground_truth,
    save_ground_truth,
)

print("Python del kernel:")
print(sys.executable)

print("\nRaíz del proyecto:")
print(PROJECT_ROOT)

print("\nDataset:")
print(DEFAULT_DATA_PATH)

## 2. Función de generación

Los parámetros principales son:

- `origin`: primer día del forecast;
- `historical_method`: `same_calendar_date` o `moving_window`;
- `context_len`: longitud del contexto entregado a TimesFM;
- `horizon_len`: cantidad de días futuros a pronosticar.

La función también permite configurar el caso de ventanas móviles.

`export_ground_truth=True` indica que, una vez terminada la generación, se escriba:

```text
ground_truth.csv
```

en la misma carpeta de resultados.

In [ ]:
def generate_wind_scenarios(
    origin: str,
    historical_method: str = "same_calendar_date",
    context_len: int = 32,
    horizon_len: int = 28,
    *,
    n_scenarios: int | None = None,
    window_radius_days: int = 15,
    min_start_distance_days: int = 0,
    selection: str = "closest",
    device: str = "auto",
    export_ground_truth: bool = True,
    data_path: str | Path = DEFAULT_DATA_PATH,
    output_root: str | Path = PROJECT_ROOT / "outputs" / "schaake",
) -> dict:
    """
    Run zero-shot TimesFM + Schaake Shuffle using the repository's
    generate module.

    Ground truth is loaded only after generation has completed.
    """
    if historical_method not in {
        "same_calendar_date",
        "moving_window",
    }:
        raise ValueError(
            "historical_method must be "
            "'same_calendar_date' or 'moving_window'."
        )

    if context_len <= 0:
        raise ValueError(
            "context_len must be positive."
        )

    if horizon_len <= 0:
        raise ValueError(
            "horizon_len must be positive."
        )

    if device not in {
        "auto",
        "cpu",
        "cuda",
    }:
        raise ValueError(
            "device must be 'auto', 'cpu' or 'cuda'."
        )

    origin_ts = pd.Timestamp(
        origin
    ).normalize()

    data_path = Path(
        data_path
    )

    output_root = Path(
        output_root
    )

    command = [
        sys.executable,
        "-m",
        "wind_scenarios.cli.generate",
        "--origin",
        origin_ts.date().isoformat(),
        "--historical_method",
        historical_method,
        "--context_len",
        str(context_len),
        "--horizon_len",
        str(horizon_len),
        "--device",
        device,
        "--data_path",
        str(data_path),
        "--output_dir",
        str(output_root),
    ]

    if n_scenarios is not None:
        command.extend(
            [
                "--n_scenarios",
                str(n_scenarios),
            ]
        )

    if historical_method == "moving_window":
        command.extend(
            [
                "--window_radius_days",
                str(window_radius_days),
                "--min_start_distance_days",
                str(min_start_distance_days),
                "--selection",
                selection,
            ]
        )

    print(
        "=== TIMESFM + SCHAAKE SHUFFLE ==="
    )
    print(
        " ".join(
            f'"{part}"'
            if " " in part
            else part
            for part in command
        )
    )
    print()

    # The subprocess must finish successfully before any future
    # observation is read for evaluation.
    subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        check=True,
    )

    output_dir = (
        output_root
        / origin_ts.date().isoformat()
    )

    ground_truth_path = None

    if export_ground_truth:
        ground_truth_path = save_ground_truth(
            origin=origin_ts,
            horizon_len=horizon_len,
            output_dir=output_dir,
            data_path=data_path,
        )

    paths = {
        "output_dir": output_dir,
        "context": (
            output_dir
            / "context.csv"
        ),
        "quantiles": (
            output_dir
            / "timesfm_quantiles.csv"
        ),
        "historical": (
            output_dir
            / "historical_trajectories.csv"
        ),
        "metadata": (
            output_dir
            / "historical_metadata.csv"
        ),
        "ranks": (
            output_dir
            / "historical_ranks.csv"
        ),
        "pseudo_observations": (
            output_dir
            / "empirical_pseudo_observations.csv"
        ),
        "assigned_probabilities": (
            output_dir
            / "assigned_probabilities.csv"
        ),
        "marginals": (
            output_dir
            / "timesfm_discretized_marginals.csv"
        ),
        "scenarios": (
            output_dir
            / "scenarios.csv"
        ),
        "ground_truth": ground_truth_path,
    }

    result = {
        "origin": origin_ts,
        "historical_method": historical_method,
        "context_len": context_len,
        "horizon_len": horizon_len,
        "output_dir": output_dir,
        "paths": paths,
        "context": pd.read_csv(
            paths["context"],
            parse_dates=["DATE"],
        ),
        "quantiles": pd.read_csv(
            paths["quantiles"],
            parse_dates=["DATE"],
        ),
        "scenarios": pd.read_csv(
            paths["scenarios"],
        ),
    }

    if ground_truth_path is not None:
        result["ground_truth"] = (
            pd.read_csv(
                ground_truth_path,
                parse_dates=["DATE"],
            )
        )

    print()
    print(
        "=== GENERACIÓN COMPLETADA ==="
    )
    print(
        f"Output:        {output_dir}"
    )
    print(
        f"Context:       {result['context'].shape}"
    )
    print(
        f"Quantiles:     {result['quantiles'].shape}"
    )
    print(
        f"Scenarios:     {result['scenarios'].shape}"
    )

    if ground_truth_path is not None:
        print(
            "Ground truth:  "
            f"{result['ground_truth'].shape}"
        )
        print(
            f"Saved:         {ground_truth_path}"
        )

    return result

## 3. Caso base: 32 días de contexto y 28 días de forecast

Equivale conceptualmente a:

```powershell
uv run generate-wind-scenarios `
    --origin 2014-04-01 `
    --historical_method same_calendar_date `
    --context_len 32 `
    --horizon_len 28
```

In [ ]:
result = generate_wind_scenarios(
    origin="2014-04-01",
    historical_method="same_calendar_date",
    context_len=32,
    horizon_len=28,
)

## 4. Revisar resultados

In [ ]:
print("Carpeta:")
print(result["output_dir"])

print("\nArchivos:")
for name, path in result["paths"].items():
    if path is not None:
        print(
            f"{name:24s} -> {path}"
        )

print("\nContexto:")
display(
    result["context"].head()
)

print("\nCuantiles TimesFM:")
display(
    result["quantiles"].head()
)

print("\nGround truth:")
display(
    result["ground_truth"].head()
)

## 5. Caso 60/60

Para aproximadamente dos meses de contexto y dos meses de forecast:

In [ ]:
# Descomenta para ejecutar.
#
# result_60 = generate_wind_scenarios(
#     origin="2014-04-01",
#     historical_method="same_calendar_date",
#     context_len=60,
#     horizon_len=60,
# )

## 6. Caso de 100 escenarios con ventanas móviles

In [ ]:
# Descomenta para ejecutar.
#
# result_100 = generate_wind_scenarios(
#     origin="2014-04-01",
#     historical_method="moving_window",
#     context_len=32,
#     horizon_len=28,
#     n_scenarios=100,
#     window_radius_days=15,
#     min_start_distance_days=7,
# )

## 7. Uso independiente de `evaluation.py`

Si ya existen resultados y sólo quieres reconstruir el target observado:

```python
ground_truth = load_ground_truth(
    origin="2014-04-01",
    horizon_len=28,
)
```

o guardarlo directamente:

```python
save_ground_truth(
    origin="2014-04-01",
    horizon_len=28,
    output_dir="outputs/schaake/2014-04-01",
)
```

`ground_truth.csv` contiene:

```text
DATE,WIND,LEAD_DAY
```

y se alinea exactamente con los `horizon_len` días del forecast.